# Normal RAG Inference (Kaggle T4 x2)

Run the dataset-agnostic normal RAG pipeline with the merged thesis dataset as the main/default run, then MS MARCO as a secondary benchmark. This uses FAISS retrieval plus `Qwen/Qwen2.5-7B-Instruct` in 4-bit causal generation.

The prompt is normal RAG and does not instruct the model to answer `I don't know`; every row should receive a direct answer.

In [ ]:
# Kaggle bootstrap. Enable GPU T4 x2 and Internet before running.
# Avoid vLLM in Kaggle's shared image to prevent CUDA/numpy/protobuf resolver conflicts.
!pip install -q accelerate bitsandbytes sentencepiece sentence-transformers faiss-cpu rouge-score

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

MERGED_CONFIG = "configs/experiments/normal_rag_merged.yaml"
MSMARCO_CONFIG = "configs/experiments/normal_rag_msmarco.yaml"
print("Project root:", PROJECT_ROOT)
print("CUDA visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES", "default"))

In [ ]:
from rag_filtering.config.loader import load_yaml

for config_path, expected_dataset_type in [
    (MERGED_CONFIG, "merged"),
    (MSMARCO_CONFIG, "msmarco"),
]:
    cfg = load_yaml(config_path)
    assert cfg["data"]["dataset_type"] == expected_dataset_type
    assert cfg["model"]["backend"] == "causal_instruct"
    assert cfg["model"]["name"] == "Qwen/Qwen2.5-7B-Instruct"
    assert cfg["model"]["load_in_4bit"] is True
    assert "I don't know" not in cfg["generation"]["prompt_template"]
    print(config_path, "OK")

In [ ]:
# Main/default run: merged thesis dataset.
!python experiments/normal_rag_inference/run_inference.py --config configs/experiments/normal_rag_merged.yaml

In [ ]:
# Secondary benchmark run: MS MARCO.
!python experiments/normal_rag_inference/run_inference.py --config configs/experiments/normal_rag_msmarco.yaml

In [ ]:
import pandas as pd

def show_run(name, results_dir):
    results_dir = PROJECT_ROOT / results_dir
    predictions_path = results_dir / "predictions.csv"
    metrics_path = results_dir / "metrics.json"
    df = pd.read_csv(predictions_path)
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(f"\n=== {name} metrics ===")
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    display(df[["id", "question", "predicted_answer"]].head())
    bad = df[
        df["predicted_answer"].fillna("").astype(str).str.strip().eq("")
        | df["predicted_answer"].fillna("").astype(str).str.contains("I don't know|cannot answer|<\\|im_|<pad>|</s>", case=False, regex=True)
    ]
    if len(bad):
        print(f"Found {len(bad)} suspicious predictions")
        display(bad[["id", "question", "predicted_answer"]].head(20))
    else:
        print("No empty, abstention, or special-token predictions found.")

show_run("Merged", "results/normal_rag/merged")
show_run("MS MARCO", "results/normal_rag/msmarco")

In [ ]:
# Zip outputs for download from Kaggle's Output panel.
!cd results && zip -r normal_rag_outputs.zip normal_rag